## Config stuff

In [3]:

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import ConnectionConfig as cc
cc.setupEnvironment()


In [4]:
spark = cc.startLocalCluster("DIM_DATE", 4)
spark.getActiveSession()

In [5]:
# EXTRACT

beginDate = '2009-01-01'
endDate = '2023-12-31'

df_SQL = spark.sql(f"select explode(sequence(to_date('{beginDate}'), to_date('{endDate}'), interval 1 day)) as calendarDate, monotonically_increasing_id() as date_SK ")


df_SQL.createOrReplaceTempView('neededDates' )

spark.sql("select * from neededDates").show()

+------------+-------+
|calendarDate|date_SK|
+------------+-------+
|  2009-01-01|      0|
|  2009-01-02|      1|
|  2009-01-03|      2|
|  2009-01-04|      3|
|  2009-01-05|      4|
|  2009-01-06|      5|
|  2009-01-07|      6|
|  2009-01-08|      7|
|  2009-01-09|      8|
|  2009-01-10|      9|
|  2009-01-11|     10|
|  2009-01-12|     11|
|  2009-01-13|     12|
|  2009-01-14|     13|
|  2009-01-15|     14|
|  2009-01-16|     15|
|  2009-01-17|     16|
|  2009-01-18|     17|
|  2009-01-19|     18|
|  2009-01-20|     19|
+------------+-------+
only showing top 20 rows



In [6]:
# TRANSFORM
dimDate = spark.sql("select date_SK, \
  CalendarDate as date, \
  year(calendarDate) AS year, \
quarter(calendarDate) as quarter, \
  month(calendarDate) as month_nr, \
  date_format(calendarDate, 'MMMM') as month_name, \
    dayofmonth(calendarDate) as day_nr, \
  date_format(calendarDate, 'EEEE') as day_name, \
  case \
    when weekday(calendarDate) < 5 then 'Y' \
    else 'N' \
  end as IsWeekDay \
from  \
  neededDates \
order by \
  calendarDate")

dimDate.show()

+-------+----------+----+-------+--------+----------+------+---------+---------+
|date_SK|      date|year|quarter|month_nr|month_name|day_nr| day_name|IsWeekDay|
+-------+----------+----+-------+--------+----------+------+---------+---------+
|      0|2009-01-01|2009|      1|       1|   January|     1| Thursday|        Y|
|      1|2009-01-02|2009|      1|       1|   January|     2|   Friday|        Y|
|      2|2009-01-03|2009|      1|       1|   January|     3| Saturday|        N|
|      3|2009-01-04|2009|      1|       1|   January|     4|   Sunday|        N|
|      4|2009-01-05|2009|      1|       1|   January|     5|   Monday|        Y|
|      5|2009-01-06|2009|      1|       1|   January|     6|  Tuesday|        Y|
|      6|2009-01-07|2009|      1|       1|   January|     7|Wednesday|        Y|
|      7|2009-01-08|2009|      1|       1|   January|     8| Thursday|        Y|
|      8|2009-01-09|2009|      1|       1|   January|     9|   Friday|        Y|
|      9|2009-01-10|2009|   

+------------+------+----+-------+--------------+
|calendarDate|dateSK|year|  month|lasyDayOfMonth|
+------------+------+----+-------+--------------+
|  2009-01-01|     0|2009|January|             N|
|  2009-01-02|     1|2009|January|             N|
|  2009-01-03|     2|2009|January|             N|
|  2009-01-04|     3|2009|January|             N|
|  2009-01-05|     4|2009|January|             N|
|  2009-01-06|     5|2009|January|             N|
|  2009-01-07|     6|2009|January|             N|
|  2009-01-08|     7|2009|January|             N|
|  2009-01-09|     8|2009|January|             N|
|  2009-01-10|     9|2009|January|             N|
|  2009-01-11|    10|2009|January|             N|
|  2009-01-12|    11|2009|January|             N|
|  2009-01-13|    12|2009|January|             N|
|  2009-01-14|    13|2009|January|             N|
|  2009-01-15|    14|2009|January|             N|
|  2009-01-16|    15|2009|January|             N|
|  2009-01-17|    16|2009|January|             N|


In [7]:
# LOAD
(dimDate.write.format("delta")
     .option("overwriteSchema", "true").mode("overwrite").saveAsTable("dimDate"))


In [20]:
spark.stop()

NameError: name 'spark' is not defined